In [1]:
import os
from dotenv import load_dotenv

print("loaded:", load_dotenv(r"API.env"))   

loaded: True


# Stage A: summary generation & LLM parsing

Choosen 18 RTLs to fine tune the prompts on to save token

## technical summary generation of the selected modules

In [2]:
#config
from pathlib import Path
import json, re, time, hashlib, os

SPEC_PDF  = Path("NEORV32-patched.pdf")
RTL_DIR   = Path("RTL_data")
OUT_DIR   = Path("summaries_tuning18"); OUT_DIR.mkdir(exist_ok=True)
CACHE_DIR = Path("rag_cache_ada002");   CACHE_DIR.mkdir(exist_ok=True)

CHUNK_SIZE, CHUNK_OVERLAP, TOP_K = 1000, 200, 20   # paper params
EMBED_MODEL = "text-embedding-ada-002"             # paper's embedder
GEN_MODEL   = "gpt-5-mini"                         # paper: GPT-5 family

# Cell 0:
QUERY_RULE = "name+filelist-role-v1"

In [3]:
from pathlib import Path

def discover_modules(rtl_dir=RTL_DIR):
    """All .vhd files in the RTL dir — the full 18-module tuning set.
    Only the filename stem (= IP module name) is used at this stage;
    RTL content is not read until Task 3 (Modular RTL Parsing)."""
    return [(p.stem, p) for p in sorted(Path(rtl_dir).glob("*.vhd"))]

modules = discover_modules()
assert len(modules) == 18, f"expected 18, found {len(modules)}"
print([s for s, _ in modules])

['neorv32_boot_rom', 'neorv32_bus', 'neorv32_cache', 'neorv32_cpu', 'neorv32_cpu_cp_cfu', 'neorv32_cpu_cp_muldiv', 'neorv32_cpu_pmp', 'neorv32_debug_dtm', 'neorv32_fifo', 'neorv32_hwspinlock', 'neorv32_imem', 'neorv32_package', 'neorv32_spi', 'neorv32_sys', 'neorv32_trng', 'neorv32_twi', 'neorv32_uart', 'neorv32_wdt']


In [4]:
#Cell 1 — PDF extraction + patch sanity gate
import fitz  # pymupdf

def extract_pages(pdf_path):
    doc = fitz.open(pdf_path)
    pages = []
    for i, page in enumerate(doc):
        txt = page.get_text("text")
        txt = re.sub(r"[ \t]+", " ", txt)
        txt = re.sub(r"\n{3,}", "\n\n", txt)
        pages.append({"page": i + 1, "text": txt})
    doc.close()
    return pages

pages = extract_pages(SPEC_PDF)
print(f"{len(pages)} pages, {sum(len(p['text']) for p in pages):,} chars")

239 pages, 419,345 chars


In [5]:
FOOTER_RE = re.compile(
    # page header
    r"The\s+NEORV32\s+RISC-V\s+Processor\s+Visit\s+on\s+GitHub"
    r"|"
    # page footer: pageno / 238 + (Version ... | Copyright ...) + date, as ONE unit
    r"\d{1,3}\s*/\s*238\s+"
    r"(?:Version\s+v1\.11\.0-r51-gd29fe6ec(?:-patched)?"
    r"|Copyright\s+by\s+Stephan\s+Nolting\.\s+All\s+rights\s+reserved\.)"
    r"\s+\d{4}-\d{2}-\d{2}"
)

pages_raw = extract_pages(SPEC_PDF)
assert any("-patched" in p["text"] for p in pages_raw), "wrong PDF: no -patched marker"

pages = [{"page": p["page"], "text": FOOTER_RE.sub(" ", p["text"]).strip()}
         for p in pages_raw]

# quick residue check — should print nothing:
for p in pages:
    if "Visit on GitHub" in p["text"] or re.search(r"/\s*238", p["text"]):
        print("footer residue on page", p["page"])

In [6]:
#cell 2— page aware chunking
def chunk_pages(pages, size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    chunks, step = [], size - overlap
    for p in pages:
        txt = p["text"].strip()
        if not txt:
            continue
        for start in range(0, max(len(txt) - overlap, 1), step):
            piece = txt[start:start + size].strip()
            if len(piece) < 100:
                continue
            chunks.append({"id": len(chunks), "page": p["page"], "text": piece})
    return chunks

EXCLUDE_PAGES = {1, 2, 3, 4, 5, 6, 7,        # cover, doc links, TOC
                 8, 12, 13,                   # ch.1 overview / marketing bullets
                 9, 10, 11,                   # §1.1 Rationale — author essay, no spec content
                 192, 193,                    # §4.2 HAL driver-file table (both pages)
                 210,                         # §4.10 RTE — software trap handling
                 237, 238, 239}               # ch.6 Legal (all three pages)                        # only pages verified as cover/TOC/index/name-lists.
                        # Do NOT exclude content pages that merely rank too often.
chunks = chunk_pages([p for p in pages if p["page"] not in EXCLUDE_PAGES])

print(f"{len(chunks)} chunks")

495 chunks


In [7]:
# Derive per-module role lines mechanically from the datasheet's own RTL File List (p.15-16)
role_src = " ".join(p["text"] for p in pages_raw if p["page"] in (15, 16))
role_src = re.sub(r"\s+", " ", role_src)
ROLE_RE = re.compile(r"neorv32_(\w+?)\.vhd\s*-\s*(.+?)(?=\s*[│├└]?\s*-?\s*neorv32_\w+\.vhd|$)")
FILE_ROLES = {f"neorv32_{m.group(1)}": m.group(2).strip(" -│├└")
              for m in ROLE_RE.finditer(role_src)}

# verify coverage for the 18 tuning modules before trusting it:
for stem, _ in modules:
    print(f"{stem:24s} -> {FILE_ROLES.get(stem, '*** MISSING ***')}")

neorv32_boot_rom         -> Bootloader ROM
neorv32_bus              -> SoC bus infrastructure modules
neorv32_cache            -> Generic cache module
neorv32_cpu              -> NEORV32 CPU TOP ENTITY
neorv32_cpu_cp_cfu       -> Custom instructions co-processor (Zxcfu ext.)
neorv32_cpu_cp_muldiv    -> Mul/Div co-processor (M ext.)
neorv32_cpu_pmp          -> Physical memory protection unit (Smpmp ext.)
neorv32_debug_dtm        -> On-chip debugger: debug transfer module
neorv32_fifo             -> Generic FIFO component
neorv32_hwspinlock       -> *** MISSING ***
neorv32_imem             -> Generic processor-internal instruction memory The NEORV32 RISC-V Processor Visit on GitHub 14 / 238 Version v1.11.0-r51-gd29fe6ec-patched 2026-07-20
neorv32_package          -> Main VHDL package file
neorv32_spi              -> Serial peripheral interface controller (SPI host)
neorv32_sys              -> System infrastructure modules
neorv32_trng             -> True random number generator
neorv32_t

In [8]:
#Cell 3 — ada-002 embeddings + FAISS (cached)
import numpy as np, faiss
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
oai = OpenAI()  # reads OPENAI_API_KEY

INDEX_PATH  = CACHE_DIR / "spec.faiss"
CHUNKS_PATH = CACHE_DIR / "chunks.json"
HASH_PATH   = CACHE_DIR / "index_hash.txt"
# Cell 3, replace the index_hash line:
CLEAN_TAG = "footerstrip-v1"   # bump whenever extraction/cleaning logic changes
chunks_digest = hashlib.sha256(
    "".join(c["text"] for c in chunks).encode()
).hexdigest()[:16]
index_hash = hashlib.sha256(
    f"{chunks_digest}|{CHUNK_SIZE}|{CHUNK_OVERLAP}|{EMBED_MODEL}|{CLEAN_TAG}".encode()
).hexdigest()[:16]

def embed_texts(texts, batch=128):
    """ada-002, batched. Returns float32 array, L2-normalized (IP == cosine)."""
    vecs = []
    for i in range(0, len(texts), batch):
        resp = oai.embeddings.create(model=EMBED_MODEL, input=texts[i:i + batch])
        vecs.extend(d.embedding for d in resp.data)
        print(f"  embedded {min(i + batch, len(texts))}/{len(texts)}")
    arr = np.asarray(vecs, dtype="float32")
    arr /= np.linalg.norm(arr, axis=1, keepdims=True)
    return arr

if INDEX_PATH.exists() and HASH_PATH.exists() and HASH_PATH.read_text() == index_hash:
    index = faiss.read_index(str(INDEX_PATH))
    chunks = json.loads(CHUNKS_PATH.read_text())
    print(f"loaded cached index ({index.ntotal} vectors)")
else:
    embs = embed_texts([c["text"] for c in chunks])
    index = faiss.IndexFlatIP(embs.shape[1])          # 1536-dim
    index.add(embs)
    faiss.write_index(index, str(INDEX_PATH))
    CHUNKS_PATH.write_text(json.dumps(chunks))
    HASH_PATH.write_text(index_hash)
    print(f"built index: {index.ntotal} vectors, hash {index_hash}")

def retrieve(query, k=TOP_K):
    q = embed_texts([query])
    scores, ids = index.search(q, k)
    return [{**chunks[i], "score": float(s)} for i, s in zip(ids[0], scores[0])]

loaded cached index (495 vectors)


In [9]:
#Cell 5 — prompts (revised: name-only identity and query)
import prompts, importlib
importlib.reload(prompts)  # for iterative dev, reload the module
from prompts import SUMMARY_SYSTEM

def build_query(stem):
    """User-guided query: module name + the datasheet's own RTL File List
    one-line description (§1.4). Uniform mechanical rule, no per-module tuning."""
    human = stem.replace("neorv32_", "").replace("_", " ")
    role = FILE_ROLES.get(stem, "")
    return f"NEORV32 {human} module {role}".strip()

def build_summary_user(stem, retrieved):
    excerpts = "\n\n".join(
        f"[Excerpt {i+1} | p.{c['page']}]\n{c['text']}"
        for i, c in enumerate(retrieved)
    )
    return (f"TARGET IP MODULE: {stem}\n\n"
            f"=== RETRIEVED SPECIFICATION EXCERPTS ===\n{excerpts}\n"
            f"=== END OF EXCERPTS ===\n\n"
            f"Produce the Technical Summary for IP module '{stem}' now, "
            f"following the required structure.")

In [10]:
for stem in ["neorv32_cpu_pmp", "neorv32_cpu", "neorv32_debug_dtm", "neorv32_package"]:
    got = sorted({c["page"] for c in retrieve(build_query(stem))})
    print(f"{stem:22s} pages={got}   CSR-region hits={[p for p in got if 160<=p<=175]}")

  embedded 1/1
neorv32_cpu_pmp        pages=[15, 21, 22, 30, 38, 44, 84, 122, 128, 139, 141, 147, 170, 171, 204, 214, 219, 234]   CSR-region hits=[170, 171]
  embedded 1/1
neorv32_cpu            pages=[15, 18, 19, 21, 22, 28, 43, 44, 84, 122, 124, 125, 131, 132, 135, 147, 214, 234]   CSR-region hits=[]
  embedded 1/1
neorv32_debug_dtm      pages=[15, 22, 84, 92, 96, 214, 215, 217, 218, 219, 228, 230, 234]   CSR-region hits=[]
  embedded 1/1
neorv32_package        pages=[14, 15, 16, 18, 19, 22, 50, 55, 78, 84, 92, 100, 107, 111, 191, 197]   CSR-region hits=[]


In [11]:
#Cell 6 — generation loop (revised: OpenAI, gpt-5-mini)
# `oai` client already created in Cell 3 for ada-002.

def generate(system, user, retries=4):
    for attempt in range(retries):
        try:
            resp = oai.chat.completions.create(
                model=GEN_MODEL,
                messages=[{"role": "system", "content": system},
                          {"role": "user", "content": user}],
                # NOTE: gpt-5-family models may reject non-default temperature;
                # omit it rather than pin it. Log the default in provenance.
            )
            return resp.choices[0].message.content
        except Exception as e:
            wait = 20 * (attempt + 1)
            print(f"  retry {attempt+1}: {e} (sleep {wait}s)")
            time.sleep(wait)
    raise RuntimeError("generation failed after retries")

for stem, _vhd_path in modules:
    out_md = OUT_DIR / f"{stem}.md"
    if out_md.exists():
        print(f"skip {stem} (exists)")
        continue
    query = build_query(stem)
    retrieved = retrieve(query)
    summary = generate(SUMMARY_SYSTEM, build_summary_user(stem, retrieved))
    out_md.write_text(summary, encoding="utf-8")
    (OUT_DIR / f"{stem}.provenance.json").write_text(json.dumps({
        "module": stem,
        "query": query,
        "query_rule": QUERY_RULE,          # <-- here
        "file_role": FILE_ROLES.get(stem, ""),          # <-- optional, see below
        "chunk_ids": [c["id"] for c in retrieved],
        "pages": sorted({c["page"] for c in retrieved}),
        "scores": [round(c["score"], 4) for c in retrieved],
        "embed_model": EMBED_MODEL, "gen_model": GEN_MODEL,
        "index_hash": index_hash, "spec": SPEC_PDF.name,
    }, indent=2), encoding="utf-8")
    print(f"done {stem}  pages={sorted({c['page'] for c in retrieved})}")

  embedded 1/1
done neorv32_boot_rom  pages=[14, 16, 22, 44, 45, 50, 191, 199, 204, 205, 206, 208, 214, 234]
  embedded 1/1
done neorv32_bus  pages=[16, 22, 36, 38, 40, 43, 55, 59, 84, 100, 111, 122, 128, 131, 132, 191, 204, 214]
  embedded 1/1
done neorv32_cache  pages=[16, 19, 21, 22, 28, 43, 51, 53, 84, 122, 131, 132, 135, 147, 150, 214, 234]
  embedded 1/1
done neorv32_cpu  pages=[15, 18, 19, 21, 22, 28, 43, 44, 84, 122, 124, 125, 131, 132, 135, 147, 214, 234]
  embedded 1/1
done neorv32_cpu_cp_cfu  pages=[21, 22, 23, 109, 122, 135, 147, 150, 151, 152, 153, 154, 180]
  embedded 1/1
done neorv32_cpu_cp_muldiv  pages=[15, 19, 21, 22, 34, 43, 76, 122, 132, 135, 139, 141, 147, 150, 204, 214, 234]
  embedded 1/1
done neorv32_cpu_pmp  pages=[15, 21, 22, 30, 38, 44, 84, 122, 128, 139, 141, 147, 170, 171, 204, 214, 219, 234]
  embedded 1/1
done neorv32_debug_dtm  pages=[15, 22, 84, 92, 96, 214, 215, 217, 218, 219, 228, 230, 234]
  embedded 1/1
done neorv32_fifo  pages=[18, 19, 22, 36, 55, 

### checkings

In [14]:
from collections import Counter

prov = [json.loads(p.read_text()) for p in sorted(OUT_DIR.glob("*.provenance.json"))]
freq = Counter(pg for r in prov for pg in r["pages"])
attractors = [pg for pg, n in freq.most_common() if n >= 8]
print("attractor pages (>=8/18 modules):", attractors)

for pg in attractors:
    txt = next(p["text"] for p in pages if p["page"] == pg)
    print(f"\n=== p.{pg} (in {freq[pg]}/18) ===\n{txt[:300]}")

attractor pages (>=8/18 modules): [22, 214, 84, 234, 122, 100, 92, 132, 21]

=== p.22 (in 17/18) ===
Chapter 2. NEORV32 Processor (SoC)
The NEORV32 Processor is build around the NEORV32 Central Processing Unit (CPU). Together with
common peripheral interfaces and embedded memories it provides a RISC-V-based full-scale
microcontroller-like SoC platform.
Figure 1. The NEORV32 Processor (Block Diagram

=== p.214 (in 15/18) ===
Chapter 5. On-Chip Debugger (OCD)
The NEORV32 Processor features an on-chip debugger (OCD) compatible to the Minimal RISC-V
Debug Specification implementing the execution-based debugging scheme. A copy of the
specification is available in docs/references. The on-chip debugger is implemented if the O

=== p.84 (in 14/18) ===
2.8.15. Serial Peripheral Interface Controller (SPI)
Hardware source files:
neorv32_spi.vhd
Software driver files:
neorv32_spi.c
Online software reference (Doxygen)
neorv32_spi.h
Online software reference (Doxygen)
Top entity ports:
spi_clk_o
1-b

In [12]:
#Cell 7 — QC, aimed at the fine-tuning objective
# Check both ends of the spectrum:
#   asset-rich modules -> section 2/5 must name real registers/CSRs
#   asset-poor modules -> section 5 must NOT be inflated
QC_RICH = ["neorv32_cpu_pmp", "neorv32_trng", "neorv32_wdt", "neorv32_fifo", "neorv32_package", "neorv32_hwspinlock"]
QC_POOR = [s for s, _ in modules if s not in QC_RICH][:3]  # eyeball a few others

for stem in QC_RICH + QC_POOR:
    f = OUT_DIR / f"{stem}.md"
    if f.exists():
        print(f"\n{'='*60}\n{stem}\n{'='*60}\n{f.read_text()[:1500]}")




neorv32_cpu_pmp
MODULE: neorv32_cpu_pmp
1. FUNCTION AND ROLE
The neorv32_cpu_pmp implements the NEORV32 physical memory protection (PMP) (Smpmp ISA Extension) and provides permission checking for instruction and data accesses. It configures read/write/execute permissions and region addressing to grant/revoke access between machine-mode and user-mode and to enforce custom physical memory attributes (PMAs) at CPU bus access time. PMP behavior and capabilities are adjusted via Processor Top Entity generics and a set of Machine Physical Memory Protection CSRs.

2. REGISTERS, CSRS, AND FLAGS
- pmpcfg* — r/w — 0x00000000 — Configuration of physical memory protection regions (pmpcfg0..pmpcfg3 at addresses 0x3a0..0x3a3). Each region provides an individual 8-bit array in these CSRs.
- PMPCFG_R — r/w — 0x0 — R: Read permission (bit field within pmpcfg*).
- PMPCFG_W — r/w — 0x0 — W: Write permission (bit field within pmpcfg*).
- PMPCFG_X — r/w — 0x0 — X: Execute permission (bit field within pmpc

In [17]:
t = (OUT_DIR / "neorv32_cpu_pmp.md").read_text()
print("D3 content present:", "31 downto 2" in t or "downto 2" in t)

D3 content present: True


In [18]:
# Registers/ports owned by module X must not appear in module Y's summary.
FOREIGN = {"neorv32_spi": "SPI_", "neorv32_twi": "TWI_", "neorv32_uart": "UART_",
           "neorv32_wdt": "WDT_", "neorv32_trng": "TRNG_",
           "neorv32_cpu_pmp": "PMPCFG_", "neorv32_cpu_cp_cfu": "cfureg"}

for md in sorted(OUT_DIR.glob("*.md")):
    text = md.read_text(encoding="utf-8")
    for owner, prefix in FOREIGN.items():
        if md.stem != owner and prefix in text:
            line = next(l for l in text.splitlines() if prefix in l)
            print(f"CHECK  {prefix!r} in {md.stem:24s} | {line.strip()}")

In [19]:
SENTINEL = "Not covered in retrieved spec sections."
LEAK_TERMS = ["excerpt", "retriev", "provided context", "the context above",
              "these instructions", "system prompt", "[Excerpt"]

leaks = []
for md in sorted(OUT_DIR.glob("*.md")):
    body = md.read_text(encoding="utf-8").replace(SENTINEL, "")  # whitelist sentinel
    low = body.lower()
    for term in LEAK_TERMS:
        if term.lower() in low:
            # show the offending line for eyeball
            line = next(l for l in body.splitlines() if term.lower() in l.lower())
            leaks.append((md.stem, term, line.strip()[:120]))

if leaks:
    for stem, term, line in leaks:
        print(f"LEAK  {stem:24s} [{term}]  {line}")
else:
    print("meta-leakage: clean (18/18)")

LEAK  neorv32_cpu_cp_muldiv    [excerpt]  The physical implementation of the multiplier can be tuned via CPU tuning options (unnamed in the excerpts) to trade are


In [20]:
import re

EXPECTED = ["1. FUNCTION AND ROLE",
            "2. REGISTERS, CSRS, AND FLAGS",
            "3. CONFIGURATION",
            "4. CROSS-MODULE INTERACTION",
            "5. SECURITY-RELEVANT BEHAVIOR"]

fails = []
for md in sorted(OUT_DIR.glob("*.md")):
    body = md.read_text(encoding="utf-8")
    if not body.lstrip().startswith(f"MODULE: {md.stem}"):
        fails.append((md.stem, "MODULE header missing or wrong name"))
    found = re.findall(r"^\s*(\d\. [A-Z][A-Z ,\-]+)\s*$", body, re.MULTILINE)
    found = [f.strip() for f in found]
    if found != EXPECTED:
        fails.append((md.stem, f"sections={found}"))
    # every section must have content or the exact sentinel — no empty sections
    for i, hdr in enumerate(EXPECTED):
        start = body.find(hdr) + len(hdr)
        end = body.find(EXPECTED[i+1]) if i+1 < len(EXPECTED) else len(body)
        if not body[start:end].strip():
            fails.append((md.stem, f"empty section: {hdr}"))

if fails:
    for stem, msg in fails:
        print(f"FAIL  {stem:24s} {msg}")
else:
    print("section structure: clean (18/18)")

section structure: clean (18/18)


In [21]:
full = "\n".join(p["text"] for p in pages_raw)   # pre-exclusion, whole document
for term in ["hwspinlock", "spinlock", "neorv32_fifo", "Hardware Spinlock"]:
    hits = [p["page"] for p in pages_raw if term.lower() in p["text"].lower()]
    print(f"{term!r:22s} pages={hits}")

'hwspinlock'           pages=[]
'spinlock'             pages=[]
'neorv32_fifo'         pages=[15]
'Hardware Spinlock'    pages=[]


In [23]:
import re, json
from pathlib import Path

full_squash = re.sub(r"\s+", "", "".join(p["text"] for p in pages_raw)).lower()
by_id = {c["id"]: c["text"] for c in chunks}

TOKEN_RES = [
    r"\b[A-Z][A-Z0-9]+(?:_[A-Z0-9]+)+\b",            # WDT_CTRL_EN, PMP_NUM_REGIONS
    r"\b0x[0-9a-fA-F]{4,}\b",                        # 0xfffb0000, 0x709D1AB3
    r"\b[a-z][a-z0-9]*(?:_[a-z0-9]+)+_[io]\b",       # spi_csn_o, rstn_wdt_o
    r"\b(?:pmpcfg|pmpaddr|cfureg)\d+\b",
    r"\b(?:misa|mxisa|mhartid|mepc|dpc|mcounteren|dtmcs|idcode)\b",
]
TOKEN_RES += [
    r"\b[a-z][a-z0-9]*(?:_[a-z0-9]+)+_[a-z]\b",   # bus_req_t, bus_rsp_t
    r"\b[id]_bus_\*",                             # i_bus_*, d_bus_*
    r"\b(?:XBUS|SYSINFO|BOOTROM|IMEM|DMEM|CLINT|CFS|CFU|SLINK|NEOLED)\b",
    r"\b[id]CACHE\b", r"\bSmpmp\b", r"\bZ[a-z]{2,}\b",   # Zicsr, Zmmul, Zxcfu
    r"\b0b[01]+\b", r"\b\d+\s?(?:kB|MB|bit|-bit)\b",
]

report = {}
for md in sorted(OUT_DIR.glob("*.md")):
    stem = md.stem
    prov = json.loads((OUT_DIR / f"{stem}.provenance.json").read_text())
    ctx = re.sub(r"\s+", "", " ".join(by_id[i] for i in prov["chunk_ids"])).lower()
    body = md.read_text(encoding="utf-8")

    toks = set()
    for rx in TOKEN_RES:
        toks |= set(re.findall(rx, body))
    grounded, ungrounded_but_real, hallucinated = [], [], []
    for t in sorted(toks):
        k = re.sub(r"\s+", "", t).lower()
        if k in ctx:                grounded.append(t)
        elif k in full_squash:      ungrounded_but_real.append(t)
        else:                       hallucinated.append(t)

    report[stem] = dict(n=len(toks), grounded=len(grounded),
                        leaked=ungrounded_but_real, halluc=hallucinated)
    flag = "" if not (ungrounded_but_real or hallucinated) else "  <-- REVIEW"
    print(f"{stem:24s} {len(grounded):3d}/{len(toks):3d} grounded{flag}")
    if ungrounded_but_real: print(f"    LEAKED (in spec, not in retrieved ctx): {ungrounded_but_real}")
    if hallucinated:        print(f"    NOT IN SPEC AT ALL:                     {hallucinated}")

tot = sum(r["n"] for r in report.values()); g = sum(r["grounded"] for r in report.values())
print(f"\ncorpus grounding rate: {g}/{tot} = {100*g/tot:.1f}%")

neorv32_boot_rom           4/  4 grounded
neorv32_bus               10/ 10 grounded
neorv32_cache             16/ 16 grounded
neorv32_cpu               22/ 23 grounded  <-- REVIEW
    LEAKED (in spec, not in retrieved ctx): ['dbus_rsp_i']
neorv32_cpu_cp_cfu         8/  8 grounded
neorv32_cpu_cp_muldiv      2/  2 grounded
neorv32_cpu_pmp           21/ 21 grounded
neorv32_debug_dtm          5/  7 grounded  <-- REVIEW
    LEAKED (in spec, not in retrieved ctx): ['1-bit']
    NOT IN SPEC AT ALL:                     ['41-bit']
neorv32_fifo               0/  0 grounded
neorv32_hwspinlock         0/  0 grounded
neorv32_imem               5/  5 grounded
neorv32_package            0/  0 grounded
neorv32_spi               12/ 12 grounded
neorv32_sys                0/  0 grounded
neorv32_trng              12/ 12 grounded
neorv32_twi                8/  8 grounded
neorv32_uart              14/ 14 grounded
neorv32_wdt               17/ 17 grounded

corpus grounding rate: 156/159 = 98.1%


In [28]:
prov = json.loads((OUT_DIR/"neorv32_cpu.provenance.json").read_text())
for i in prov["chunk_ids"]:
    if "dbus" in by_id[i].lower():
        print(i, [w for w in by_id[i].split() if "dbus" in w.lower()])

248 ['dbus_req_o']


In [31]:
from pathlib import Path
import re

A = {p.stem for p in Path("RTL_data").glob("*.vhd")}                 # LAsset RTL (51)
RAW_57 = """
$NEORV32_HOME/rtl/core/neorv32_package.vhd
...paste all 57 lines...
"""
B = {Path(l.strip()).stem for l in RAW_57.strip().splitlines() if l.strip()}
C = set(re.findall(r"(neorv32_\w+)\.vhd",                            # datasheet §1.4 @ d29fe6ec
                   " ".join(p["text"] for p in pages_raw if p["page"] in (15, 16))))

print(f"A LAsset RTL_data      : {len(A)}")
print(f"B pasted list          : {len(B)}")
print(f"C datasheet @ d29fe6ec : {len(C)}\n")

def show(label, s):
    print(f"{label} ({len(s)}):")
    for x in sorted(s): print("   ", x)
    print()

show("A \\ C  — in your RTL, spec silent  [THE ONE THAT MATTERS]", A - C)
show("C \\ A  — spec documents, RTL absent", C - A)
show("A \\ B", A - B)
show("B \\ A", B - A)

# rename detection across naming conventions (boot_rom vs bootrom, cp_ vs alu_)
def norm(s): return s.replace("_", "").lower()
nA, nB = {norm(x): x for x in A}, {norm(x): x for x in B}
for k in set(nA) & set(nB):
    if nA[k] != nB[k]: print(f"RENAME: {nA[k]} <-> {nB[k]}")

# version stamp carried in the RTL itself
pkg = Path("RTL_data/neorv32_package.vhd").read_text(errors="replace")
for line in pkg.splitlines():
    if re.search(r"(?i)version", line): print("PKG:", line.strip())

A LAsset RTL_data      : 18
B pasted list          : 2
C datasheet @ d29fe6ec : 49

A \ C  — in your RTL, spec silent  [THE ONE THAT MATTERS] (1):
    neorv32_hwspinlock

C \ A  — spec documents, RTL absent (32):
    neorv32_application_image
    neorv32_bootloader_image
    neorv32_cfs
    neorv32_clint
    neorv32_clockgate
    neorv32_cpu_alu
    neorv32_cpu_control
    neorv32_cpu_cp_bitmanip
    neorv32_cpu_cp_cond
    neorv32_cpu_cp_crypto
    neorv32_cpu_cp_fpu
    neorv32_cpu_cp_shifter
    neorv32_cpu_decompressor
    neorv32_cpu_icc
    neorv32_cpu_lsu
    neorv32_cpu_regfile
    neorv32_crc
    neorv32_debug_auth
    neorv32_debug_dm
    neorv32_dma
    neorv32_dmem
    neorv32_gpio
    neorv32_gptmr
    neorv32_neoled
    neorv32_onewire
    neorv32_pwm
    neorv32_sdi
    neorv32_slink
    neorv32_sysinfo
    neorv32_top
    neorv32_twd
    neorv32_xbus

A \ B (17):
    neorv32_boot_rom
    neorv32_bus
    neorv32_cache
    neorv32_cpu
    neorv32_cpu_cp_cfu
    neorv32_cp

## RTL parsing

In [11]:
# Cell 3 — the client (OpenAI; OPENAI_API_KEY in env)
import os, time, random, json
from openai import OpenAI

class LLMClient:
    def __init__(self, model=GEN_MODEL, effort="medium", max_retries=6):
        self.c = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
        self.model, self.effort, self.r = model, effort, max_retries
        self.usage = []            # per-call token accounting for cost logging

    def _call(self, system, user, json_mode, max_tokens, effort):
        kw = dict(
            model=self.model,
            instructions=system,
            input=user,
            reasoning={"effort": effort or self.effort},
            max_output_tokens=max_tokens,
        )
        if json_mode:
            kw["text"] = {"format": {"type": "json_object"}}

        wait = 2.0
        for a in range(self.r):
            try:
                r = self.c.responses.create(**kw)
                if getattr(r, "status", None) == "incomplete":
                    # ran out of max_output_tokens (reasoning shares the budget):
                    # surface it, return the (possibly truncated) text and let the
                    # caller's escalation path retry with a bigger budget.
                    why = getattr(getattr(r, "incomplete_details", None), "reason", "?")
                    print(f"    [llm warn] incomplete response ({why}) at max_output_tokens={max_tokens}")
                u = r.usage
                self.usage.append({
                    "model": self.model,
                    "in": u.input_tokens,
                    "out": u.output_tokens,
                    "reasoning": u.output_tokens_details.reasoning_tokens,
                    "cached_in": getattr(u.input_tokens_details, "cached_tokens", 0),
                })
                return r.output_text
            except Exception as e:
                if a == self.r - 1:
                    raise
                time.sleep(wait + random.uniform(0, 1)); wait *= 2

    def complete(self, system, user, max_tokens=4096, effort=None):
        """JSON-mode call (Stage A batches, asset generation)."""
        return self._call(system, user, True, max_tokens, effort)

    def complete_text(self, system, user, max_tokens=4096, effort=None):
        """Plain-text call (markdown technical summaries)."""
        return self._call(system, user, False, max_tokens, effort)

    def cost(self, price_in=0.25, price_out=2.00, price_cached=0.025):
        """USD for calls so far, at per-1M-token prices (default gpt-5-mini)."""
        i = sum(u["in"] - u["cached_in"] for u in self.usage)
        c = sum(u["cached_in"] for u in self.usage)
        o = sum(u["out"] for u in self.usage)
        rt = sum(u["reasoning"] for u in self.usage)
        total = (i*price_in + c*price_cached + o*price_out) / 1e6
        return {"calls": len(self.usage), "in": i, "cached_in": c,
                "out": o, "reasoning": rt, "usd": round(total, 4)}

client = LLMClient()

In [12]:
# ===================== Cell — Modular RTL Parsing (Prm_m) =====================
import importlib, prompts, rtl_parse
importlib.reload(prompts); importlib.reload(rtl_parse)
from prompts import PARSE_PORTS_ANNOTATE_SYSTEM, PARSE_SIGNALS_ANNOTATE_SYSTEM
from rtl_parse import build_record_registry, parse_rtl_file

# package + all modules -> so bus_req_t / bus_rsp_t (declared in neorv32_package) are known everywhere
RECORDS = build_record_registry(sorted(RTL_DIR.glob("*.vhd")))
print(f"{len(RECORDS)} record types; bus_req_t={'bus_req_t' in RECORDS}, bus_rsp_t={'bus_rsp_t' in RECORDS}")

PARSE_DIR = Path("parsed_tuning18"); PARSE_DIR.mkdir(exist_ok=True)

def _loads(txt):
    try:
        return json.loads(txt)
    except Exception:
        pass
    m = re.search(r"\{.*\}", txt, re.S)   # salvage a complete object if the tail was cut
    if m:
        try:
            return json.loads(m.group(0))
        except Exception:
            pass
    return {}                              # give up gracefully rather than raise


28 record types; bus_req_t=True, bus_rsp_t=True


In [13]:
def _annotate(system, stem, entity, rtl, elems, key, batch=50, max_tokens=8192):
    """Annotate in batches so large record-expanded port lists (e.g. bus io_switch = 596
    ports) don't overflow the model's output and truncate the JSON. A batch that still
    fails to parse degrades its elements to 'unclear' instead of crashing the module."""
    ann = {}
    for i in range(0, len(elems), batch):
        chunk = elems[i:i + batch]
        got = _loads(client.complete(system, _annot_user(stem, entity, rtl, chunk, key),
                                     max_tokens=max_tokens)).get(key, [])
        for a in got:
            if "name" in a:
                ann[a["name"]] = a
        if not got:
            print(f"    [annot warn] {stem}/{entity} {key} batch {i//batch}: "
                  f"{len(chunk)} elems left unclear")
    return ann

In [14]:
def _annot_user(stem, entity, rtl, elems, key):
    return (f"MODULE: {stem}\nENTITY: {entity}\n\n"
            f"AUTHORITATIVE {key.upper()} of entity '{entity}' (ground truth from RTL — annotate "
            f"all, add/drop none; names include record fields written verbatim with a dot, "
            f"e.g. 'ctrl.buf_req', 'host_req_i.stb'):\n{json.dumps(elems, indent=2)}\n\n"
            f"VHDL SOURCE:\n{rtl}\n\n"
            f"Return the annotated {key} as a JSON object using the required schema.")

def annotate_ports(stem, entity, rtl, ports):
    a = _annotate(PARSE_PORTS_ANNOTATE_SYSTEM, stem, entity, rtl, ports, "ports")
    return {n: v.get("function", "unclear from RTL") for n, v in a.items()}

def annotate_signals(stem, entity, rtl, signals):
    a = _annotate(PARSE_SIGNALS_ANNOTATE_SYSTEM, stem, entity, rtl, signals, "signals")
    return {n: {"kind": v.get("kind", "signal"),
                "function": v.get("function", "unclear from RTL")} for n, v in a.items()}

def parse_module(stem, path, records, overwrite=False):
    outp = PARSE_DIR / f"{stem}.json"
    if outp.exists() and not overwrite:
        return json.loads(outp.read_text())
    d = parse_rtl_file(path, records=records)
    rtl = Path(path).read_text(encoding="utf-8", errors="ignore")
    ports_all, signals_all = [], []
    for ent in d["entities"]:                       # annotate per entity -> no cross-entity name clashes
        en = ent["entity"]
        pf = annotate_ports(stem, en, rtl, ent["ports"])
        sf = annotate_signals(stem, en, rtl, ent["signals"])
        for p in ent["ports"]:
            ports_all.append({"entity": en, **p, "function": pf.get(p["name"], "unclear from RTL")})
        for s in ent["signals"]:
            signals_all.append({"entity": en, **s,
                                "kind": sf.get(s["name"], {}).get("kind", "signal"),
                                "function": sf.get(s["name"], {}).get("function", "unclear from RTL")})
    parsed = {"module": stem, "entities": [e["entity"] for e in d["entities"]],
              "ports": ports_all, "signals": signals_all}
    outp.write_text(json.dumps(parsed, indent=2))
    return parsed

parsed_all = {}
for stem, path in modules:
    try:
        parsed_all[stem] = parse_module(stem, path, RECORDS)
        p = parsed_all[stem]
        print(f"[ok  ] {stem:24s} entities={p['entities']}  {len(p['ports'])}p / {len(p['signals'])}s")
    except Exception as e:
        print(f"[err ] {stem:24s} {e}")

[ok  ] neorv32_boot_rom         entities=['neorv32_boot_rom']  19p / 2s
[ok  ] neorv32_bus              entities=['neorv32_bus_switch', 'neorv32_bus_reg', 'neorv32_bus_gateway', 'neorv32_bus_io_switch', 'neorv32_bus_amo_rmw', 'neorv32_bus_amo_rvs']  811p / 59s
[ok  ] neorv32_cache            entities=['neorv32_cache', 'neorv32_cache_memory']  46p / 40s
[ok  ] neorv32_cpu              entities=['neorv32_cpu']  49p / 66s
[ok  ] neorv32_cpu_cp_cfu       entities=['neorv32_cpu_cp_cfu']  16p / 13s
[ok  ] neorv32_cpu_cp_muldiv    entities=['neorv32_cpu_cp_muldiv']  46p / 24s
[ok  ] neorv32_cpu_pmp          entities=['neorv32_cpu_pmp']  45p / 17s
[ok  ] neorv32_debug_dtm        entities=['neorv32_debug_dtm']  13p / 29s
[ok  ] neorv32_fifo             entities=['neorv32_fifo']  11p / 16s
[ok  ] neorv32_hwspinlock       entities=['neorv32_hwspinlock']  19p / 2s
[ok  ] neorv32_imem             entities=['neorv32_imem']  19p / 8s
[ok  ] neorv32_package          entities=[]  0p / 0s
[ok  ] neorv32

# Stage B: Asset generation

In [15]:
import importlib, prompts
importlib.reload(prompts)
from prompts import ASSET_PRIMARY_SYSTEM   # ASSET_PRIMARY_CORE + spliced ICL case studies (Alg.1 line 5: ICLASSET)
ASSET_DIR = Path("assets_tuning18"); ASSET_DIR.mkdir(exist_ok=True)

def build_asset_user(stem, summary, parsed, rtl):
    return (f"TARGET IP MODULE: {stem}\n\n"
            f"=== TECHNICAL SUMMARY ===\n{summary}\n\n"
            f"=== PARSED I/O PORTS (JSON) ===\n{json.dumps(parsed['ports'], indent=1)}\n\n"
            f"=== PARSED INTERNAL SIGNALS (JSON) ===\n{json.dumps(parsed['signals'], indent=1)}\n\n"
            f"=== RTL ===\n{rtl}\n\n"
            f"Identify the primary security assets for '{stem}' and return the JSON object per the contract.")

def generate_assets(stem, path, parsed, overwrite=False):
    """Cache ONLY parse-valid results. A truncated/unparseable response is retried
    once with a doubled output budget, then raised -- never written to disk, so a
    failure can't masquerade as a genuine 'no assets' verdict."""
    outp = ASSET_DIR / f"{stem}.json"
    if outp.exists() and not overwrite:
        return json.loads(outp.read_text())
    summary = (OUT_DIR / f"{stem}.md").read_text(encoding="utf-8")
    rtl = Path(path).read_text(encoding="utf-8", errors="ignore")
    user = build_asset_user(stem, summary, parsed, rtl)
    obj = None
    for mt in (16384, 32768):          # reasoning shares the output budget at effort=high
        obj = _loads(client.complete(ASSET_PRIMARY_SYSTEM, user, max_tokens=mt, effort="high"))
        if isinstance(obj, dict) and isinstance(obj.get("Assets"), list):
            break
    if not (isinstance(obj, dict) and isinstance(obj.get("Assets"), list)):
        raise RuntimeError(f"{stem}: unparseable asset response -- nothing cached")
    result = {"IP": stem, "Assets": obj["Assets"]}
    outp.write_text(json.dumps(result, indent=2))
    return result

def validate_primary(result, parsed):
    """Keyed by (entity, name): duplicate names across a file's entities (e.g. clk_i
    in neorv32_trng/neoTRNG/neoTRNG_cell) must not yield false Entity mismatches."""
    elems = parsed["ports"] + parsed["signals"]
    keys  = {(e["entity"], e["name"]) for e in elems}
    ents  = {}
    for e in elems:
        ents.setdefault(e["name"], set()).add(e["entity"])
    OBJ = {"Confidentiality", "Integrity", "Availability"}
    issues = []
    for a in result.get("Assets", []):
        r, en = a.get("Asset RTL"), a.get("Entity")
        if r not in ents:
            issues.append(f"ungrounded Asset RTL '{r}'")
        elif (en, r) not in keys:
            issues.append(f"Entity mismatch for '{r}': '{en}' not in {sorted(ents[r])}")
        if a.get("Security Objective") not in OBJ:
            issues.append(f"bad objective for '{r}': '{a.get('Security Objective')}'")
    return issues

In [16]:
# ===== Stage B run loop: generate -> validate -> one corrective retry =====
asset_results, unresolved = {}, {}
for stem, path in modules:
    try:
        res = generate_assets(stem, path, parsed_all[stem])
        issues = validate_primary(res, parsed_all[stem])
        if issues:
            print(f"[warn] {stem}: {len(issues)} issue(s) -> one regeneration\n        "
                  + "\n        ".join(issues))
            res2 = generate_assets(stem, path, parsed_all[stem], overwrite=True)
            issues2 = validate_primary(res2, parsed_all[stem])
            if len(issues2) <= len(issues):
                res, issues = res2, issues2
            else:   # first attempt was better -- restore it to the cache
                (ASSET_DIR / f"{stem}.json").write_text(json.dumps(res, indent=2))
        asset_results[stem] = res
        if issues:
            unresolved[stem] = issues
        print(f"[ok  ] {stem:24s} {len(res['Assets']):2d} assets"
              + (f"  UNRESOLVED: {issues}" if issues else ""))
    except Exception as e:
        print(f"[err ] {stem:24s} {e}")
print("\nunresolved validation issues:", unresolved or "none")
print(client.cost())

[ok  ] neorv32_boot_rom          4 assets
[ok  ] neorv32_bus              14 assets
[ok  ] neorv32_cache            29 assets
[ok  ] neorv32_cpu              22 assets
[ok  ] neorv32_cpu_cp_cfu       20 assets
[ok  ] neorv32_cpu_cp_muldiv    16 assets
[ok  ] neorv32_cpu_pmp          12 assets
[ok  ] neorv32_debug_dtm        18 assets
[ok  ] neorv32_fifo             18 assets
[ok  ] neorv32_hwspinlock        6 assets
[ok  ] neorv32_imem             14 assets
[ok  ] neorv32_package           0 assets
[ok  ] neorv32_spi              13 assets
[ok  ] neorv32_sys              15 assets
[ok  ] neorv32_trng             17 assets
[ok  ] neorv32_twi              17 assets
[ok  ] neorv32_uart             17 assets
[ok  ] neorv32_wdt              12 assets

unresolved validation issues: none
{'calls': 0, 'in': 0, 'cached_in': 0, 'out': 0, 'reasoning': 0, 'usd': 0.0}


# Stage C: Evaluation

Score the generated asset list against two references:

1. **Manual ground truth** — sheet `Assets (Manual)` of the LAsset release's
   `Asset_Dataset_Statistics_NEORV32.xlsx`. This is the paper's *True Assets* column
   (302 elements over 41 IPs) and is the reference to optimise against.
2. **The LAsset paper's own output** — `asset_list_neorv32_initial.json` is the *same
   algorithmic stage as ours* (Alg. 1 line 5), so it is the fair head-to-head.
   `..._refined.json` is after the refinement agent (Alg. 1 lines 7–9), which this
   notebook does not implement.

Metrics are TP / FP / FN + precision / recall / F1, matching the paper's own Statistics
sheet. There is deliberately **no TN or FPR column**: the negative class ("every parsed
element that is not an asset") outnumbers the positives roughly 14:1, so FPR looks
excellent while precision is poor. Security-objective correctness is reported
**separately** from asset-set correctness, because the two move independently.

`neorv32_boot_rom`, `neorv32_fifo` and `neorv32_package` have no reference entry — the
paper pruned them at its Design Modules Extraction step (Alg. 1 line 1) *before* any
asset generation, so they are reported as *unscored* rather than counted as
all-false-positive.

> Note on recall: the refinement stage only *removes* assets (the paper's own figures:
> precision +0.083, recall −0.026). Generation-stage recall is therefore a hard ceiling
> on the final pipeline — an asset missed here can never be recovered later.

In [ ]:
# ===== Build the reference sets from the LAsset release (run once; re-run if the xlsx changes)
import importlib, gt_extract
importlib.reload(gt_extract)

LASSET_ROOT = Path(r"E:\jobs\ff\test\LAsset-Security-Assets")   # <-- the published results repo
GT_DIR      = Path("ground_truth")

gt_extract.main(LASSET_ROOT, GT_DIR)

# Sanity: the extracted counts must reconcile with the paper's own Statistics sheet
# (302 True Assets / 359 initial / 331 refined). A mismatch means the sheet layout
# changed and IP_LABEL_TO_STEM or split_elements() needs revisiting.
for f in sorted(GT_DIR.glob("*.json")):
    d = json.loads(f.read_text(encoding="utf-8"))
    n = sum(len(m["assets"]) for m in d["modules"].values())
    print(f"  {f.name:28s} {len(d['modules']):3d} modules, {n:4d} elements")

In [ ]:
# ===== Headline evaluation =====
import importlib, eval_assets as EV
importlib.reload(EV)

MANUAL_GT      = EV.load_reference(GT_DIR / "manual_gt_neorv32.json")
LASSET_INITIAL = EV.load_reference(GT_DIR / "lasset_initial.json")   # same stage as ours
LASSET_REFINED = EV.load_reference(GT_DIR / "lasset_refined.json")   # after Alg. lines 7-9

ours_vs_gt = EV.evaluate(ASSET_DIR, MANUAL_GT, parsed_dir=PARSE_DIR)
EV.report(ours_vs_gt, "OURS vs manual ground truth")

In [ ]:
# ===== Three-way comparison + objective breakdown =====
# The third row is the reference baseline: how the PAPER's own generation stage scores
# against the same manual GT on the same modules. That is the number to close on.
OUR_MODULES = set(EV.load_predictions(ASSET_DIR))

paper_as_pred = {m: {e: (objs[0] if objs else "") for e, objs in elems.items()}
                 for m, elems in LASSET_INITIAL.items()}

EV.compare([
    ("ours vs manual GT",           ours_vs_gt),
    ("ours vs LAsset initial",      EV.evaluate(ASSET_DIR, LASSET_INITIAL, parsed_dir=PARSE_DIR)),
    ("ours vs LAsset refined",      EV.evaluate(ASSET_DIR, LASSET_REFINED, parsed_dir=PARSE_DIR)),
    ("LAsset initial vs manual GT", EV.evaluate(paper_as_pred, MANUAL_GT,
                                                parsed_dir=PARSE_DIR, modules=OUR_MODULES)),
])

print("\nSecurity-objective confusion (ours vs manual GT):")
EV.objective_table(ours_vs_gt)

In [ ]:
# ===== Error inspection: exactly which elements we missed and which we invented =====
EV.report(ours_vs_gt, "OURS vs manual GT -- with names", show_names=True)

# Per-module table (pandas if available) -- handy for tracking across prompt versions.
df = EV.to_dataframe(ours_vs_gt)
if df is not None:
    display(df.sort_values("f1"))

In [ ]:
# ===== A/B a prompt version =====
# Point ASSET_DIR at a different output folder, regenerate, and compare. Keeping each
# version in its own folder means nothing is overwritten and every run stays scoreable.
#
#   ASSET_DIR = Path("assets_tuning18_v2"); ASSET_DIR.mkdir(exist_ok=True)
#   <re-run the Stage B loop>
#
# Then add it to the table:

VERSIONS = {
    "baseline": Path("assets_tuning18"),
    # "v2": Path("assets_tuning18_v2"),
}

rows = []
for label, d in VERSIONS.items():
    if d.exists() and any(d.glob("*.json")):
        rows.append((f"{label} vs manual GT",
                     EV.evaluate(d, MANUAL_GT, parsed_dir=PARSE_DIR)))
if rows:
    EV.compare(rows)